In [1]:
import pandas as pd
import os
import numpy as np
from scipy.stats import spearmanr, ttest_ind, pearsonr

In [2]:
def magnitude_norm(data, data_min=0):
    data_max = np.max(np.abs(data))
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def get_aa_counts(selected_aligned_seq):
    AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
    N_AA = len(AA_LIST)
    aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
    idx_to_aa = {v: k for k, v in aa_to_idx.items()}
    aa_idx_array = np.vectorize(aa_to_idx.get, otypes=[int])(selected_aligned_seq)
    alignments = np.zeros([aa_idx_array.shape[1], N_AA])
    for i in range(aa_idx_array.shape[1]):
        unique, counts = np.unique(aa_idx_array[:,i], return_counts=True)
        alignments[i, unique] = counts
    return alignments

def calculate_attribution_frequency(
    target_class_label: int,
    attribution_arr: np.ndarray,
    all_labels: np.ndarray,
    all_aligned_sequences: np.ndarray,
    get_aa_counts_func, # Function handle
    aa_to_idx_map: dict,
    L_SEQ: int,
    N_AA: int,
    N_MODEL: int,
    cutoff_1: float,
    model_consistency: int
) -> (np.ndarray, np.ndarray):

    # 1. Filter data based on class label
    class_mask = (all_labels == target_class_label)
    
    # 2. Get class-specific data
    class_arr = attribution_arr[:, class_mask]
    class_aligned_seq = all_aligned_sequences[class_mask]
    class_aa_counts = get_aa_counts_func(class_aligned_seq)

    # Handle case with no sequences for the class
    if class_aa_counts.sum() == 0:
        print(f"Warning: No sequences found for class {target_class_label}.")
        return np.zeros([L_SEQ, N_AA]), np.zeros([L_SEQ, N_AA], dtype=int)

    # 3. Initialize count matrix
    top_aa_attr_count = np.zeros([L_SEQ, N_AA], dtype=int)
    
    # 4. Convert sequences to indices
    vectorized_aa_to_idx = np.vectorize(aa_to_idx_map.get, otypes=[int])
    aligned_seq_aa_idx = vectorized_aa_to_idx(class_aligned_seq)

    # 5. Loop through sequences for this class
    for i, seq in enumerate(aligned_seq_aa_idx):
        respos_lst = []
        
        # 6. Loop through models
        for n in range(N_MODEL):
            current_seq_attrs = class_arr[n, i]
            
            # 7. Find responsible positions (THE KEY DIFFERENCE)
            if target_class_label == 0:
                # Find smallest values (e.g., bottom 25% if cutoff_1=75)
                percentile_val = np.percentile(current_seq_attrs, 100 - cutoff_1)
                respos = np.where(current_seq_attrs < percentile_val)[0]
            else: 
                # Find largest values (e.g., top 25% if cutoff_1=75)
                percentile_val = np.percentile(current_seq_attrs, cutoff_1)
                respos = np.where(current_seq_attrs > percentile_val)[0]
                
            respos_lst.append(respos)
        
        # 8. Find consistent positions across models
        if not respos_lst:
            continue
            
        val, count = np.unique(np.concatenate(respos_lst), return_counts=True)
        respos_consistent = val[count >= model_consistency]
        
        # 9. Get AA indices at these positions
        if respos_consistent.size > 0:
            aa_idx = seq[respos_consistent]
            
            # 10. Increment counts
            top_aa_attr_count[respos_consistent, aa_idx] += 1

    # 11. Calculate frequency
    freq = np.zeros_like(top_aa_attr_count, dtype=float)
    freq = np.divide(top_aa_attr_count, class_aa_counts, out=freq, where=class_aa_counts != 0)
    
    return freq, top_aa_attr_count

def print_top_k_attributions(class_name, attr_freq, counts, idx_to_aa_map, res_array, top_k=10):
    print(f"--- Top {top_k} Attributions for {class_name} ---")
    
    flat_freq = attr_freq.flatten()
    flat_counts = counts.flatten()
    sorted_indices = np.lexsort((flat_counts, flat_freq))
    top_k_flat_indices = sorted_indices[-top_k:][::-1]
    top_k_indices = np.unravel_index(top_k_flat_indices, attr_freq.shape)
    res_indices = top_k_indices[0]
    aa_indices = top_k_indices[1]
    top_aa = [idx_to_aa_map[aa_idx] for aa_idx in aa_indices]
    top_freqs = attr_freq[res_indices, aa_indices]
    top_counts = counts[res_indices, aa_indices]
    top_resnos = res_array[res_indices]

    print(f"{'Res #':<6} | {'AA':<3} | {'Frequency':<10} | {'Count':<5}")
    print("-" * 35)
    for resno, aa, f, count in zip(top_resnos, top_aa, top_freqs, top_counts):
        print(f"{resno:<6} | {aa:<3} | {f:<10.4f} | {count:<5}")
    print("\n")

def get_internal_correlation_pairs(samples):
    if samples.shape[0] < 2:
        return np.array([]) 
    result_matrix = spearmanr(samples, axis=1)
    if result_matrix.correlation.ndim == 0:
        return np.array([result_matrix.correlation])
    correlation_matrix = result_matrix.correlation
    n = correlation_matrix.shape[0]
    i_upper, j_upper = np.triu_indices(n, k=1)
    return correlation_matrix[i_upper, j_upper]

In [3]:
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
attr_dir = "results/attribution_maps"
data_dir = "data"
task_type = "classification"
info_df = pd.read_csv(os.path.join(root_dir, data_dir, "input_info_VRC01_IC80.csv"))
info_df.sort_values(by=["Subtype_Label", "CRF", "Subtype", "Seq_name"], inplace=True)
aligned_sequences = np.array([list(seq) for seq in info_df['Sequence']])
resno_df = pd.read_csv(os.path.join(root_dir, data_dir, "selected_residues_IC80.csv"))
resno_array = resno_df["ResLabel"].values

AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
N_AA = len(AA_LIST)
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
idx_to_aa = {v: k for k, v in aa_to_idx.items()}

In [4]:
N_SEQ, L_SEQ = len(info_df), aligned_sequences.shape[1]
N_MODEL = len(selected_models)

all_attr_array_0 = np.zeros([N_MODEL, N_SEQ, L_SEQ])
all_attr_array_1 = np.zeros([N_MODEL, N_SEQ, L_SEQ])
variant_idx = []
for n, model_name in enumerate(selected_models):
    for i, no in enumerate(info_df['Seq_no']):
        attr_0 = np.load(os.path.join(root_dir, attr_dir, task_type, model_name, str(no)+"_attribution_ig_class_0.npy"))
        all_attr_array_0[n,i] = magnitude_norm(attr_0)

        attr_1 = np.load(os.path.join(root_dir, attr_dir, task_type, model_name, str(no)+"_attribution_ig_class_1.npy"))
        all_attr_array_1[n,i] = magnitude_norm(attr_1)

In [211]:
arr_dict = {
    "Class 1 Attribution": all_attr_array_1,
    "Class 0 Attribution": all_attr_array_0,
}

for name, arr in arr_dict.items():
    print(name)
    reshaped_array = arr.reshape(N_MODEL, -1)
    corr, pvalue = spearmanr(reshaped_array, axis=1)
    print("Correlation Matrix:\n", np.round(corr, 2))
    print("Mean off-diag corr:", np.mean(corr[np.triu_indices_from(corr, k=1)]).round(2))
    print("-------")

attr_1_array = arr_dict["Class 1 Attribution"].copy()
attr_0_array = arr_dict["Class 0 Attribution"].copy()

attr_1_flat = attr_1_array.flatten()
attr_0_flat = attr_0_array.flatten()

corr, p_value = pearsonr(attr_1_flat, attr_0_flat)

print(f"Pearson Correlation of Class 1 and Class 0 Attributions:\n{corr:.4f} (P-value: {p_value})")

Class 1 Attribution
Correlation Matrix:
 [[1.   0.39 0.5  0.52 0.55]
 [0.39 1.   0.46 0.41 0.42]
 [0.5  0.46 1.   0.47 0.49]
 [0.52 0.41 0.47 1.   0.48]
 [0.55 0.42 0.49 0.48 1.  ]]
Mean off-diag corr: 0.47
-------
Class 0 Attribution
Correlation Matrix:
 [[1.   0.41 0.5  0.54 0.56]
 [0.41 1.   0.45 0.41 0.42]
 [0.5  0.45 1.   0.46 0.5 ]
 [0.54 0.41 0.46 1.   0.49]
 [0.56 0.42 0.5  0.49 1.  ]]
Mean off-diag corr: 0.47
-------
Pearson Correlation of Class 1 and Class 0 Attributions:
-0.9909 (P-value: 0.0)


In [247]:
k_values_to_test = np.concatenate([[1, 5], range(10, 51, 10)])
results_store = {name: [] for name in arr_dict.keys()}
for k in k_values_to_test:
    for name, arr_list in arr_dict.items():
        arr_stacked = np.stack(arr_list, axis=0)
        mean_abs_attr = np.mean(np.abs(arr_stacked), axis=0)
        mask = np.zeros_like(mean_abs_attr, dtype=bool)
        safe_k = min(k, L_SEQ)
        for i in range(N_SEQ):
            topk_idx = np.argpartition(mean_abs_attr[i], -safe_k)[-safe_k:]
            mask[i, topk_idx] = True
        
        masked_arr = arr_stacked[:, mask].reshape(N_MODEL, -1)
        corr, _ = spearmanr(masked_arr, axis=1)
        mean_corr = np.mean(corr[np.triu_indices_from(corr, k=1)]).round(3)

        results_store[name].append((k, mean_corr))

# --- Summary ---
for name, results in results_store.items():
    print(f"\n**{name}**")
    print(" k  | Mean Corr")
    print("----|-----------")
    for k_val, corr_val in results:
        print(f"{k_val:<3} | {corr_val:<.3f}")


**Class 1 Attribution**
 k  | Mean Corr
----|-----------
1   | 0.651
5   | 0.590
10  | 0.562
20  | 0.541
30  | 0.538
40  | 0.538
50  | 0.536

**Class 0 Attribution**
 k  | Mean Corr
----|-----------
1   | 0.666
5   | 0.608
10  | 0.583
20  | 0.556
30  | 0.547
40  | 0.544
50  | 0.543


In [30]:
cutoff_1 = 75
cutoff_2 = 0.8
model_consistency = 2

class_label = info_df["Label"].values
# select Class 1 Attribution for down stream analysis
attribution_arr = arr_dict["Class 1 Attribution"].astype(float).copy()

# Call the function for Class 0
class_0_attr_freq, class_0_counts = calculate_attribution_frequency(
    target_class_label=0,
    attribution_arr=attribution_arr,
    all_labels=class_label,
    all_aligned_sequences=aligned_sequences,
    get_aa_counts_func=get_aa_counts,
    aa_to_idx_map=aa_to_idx,
    L_SEQ=L_SEQ,
    N_AA=N_AA,
    N_MODEL=N_MODEL,
    cutoff_1=cutoff_1,
    model_consistency=model_consistency
)

# Call the function for Class 1
class_1_attr_freq, class_1_counts = calculate_attribution_frequency(
    target_class_label=1,
    attribution_arr=attribution_arr,
    all_labels=class_label,
    all_aligned_sequences=aligned_sequences,
    get_aa_counts_func=get_aa_counts,
    aa_to_idx_map=aa_to_idx,
    L_SEQ=L_SEQ,
    N_AA=N_AA,
    N_MODEL=N_MODEL,
    cutoff_1=cutoff_1,
    model_consistency=model_consistency
)

print_top_k_attributions(
    class_name="Class 0",
    attr_freq=class_0_attr_freq,
    counts=class_0_counts,
    idx_to_aa_map=idx_to_aa,
    res_array=resno_array, top_k=6
)

print_top_k_attributions(
    class_name="Class 1",
    attr_freq=class_1_attr_freq,
    counts=class_1_counts,
    idx_to_aa_map=idx_to_aa,
    res_array=resno_array, top_k=6
)

--- Top 6 Attributions for Class 0 ---
Res #  | AA  | Frequency  | Count
-----------------------------------
303    | T   | 1.0000     | 570  
198    | T   | 1.0000     | 523  
236    | T   | 1.0000     | 484  
283    | T   | 1.0000     | 430  
173    | Y   | 1.0000     | 414  
278    | T   | 1.0000     | 408  


--- Top 6 Attributions for Class 1 ---
Res #  | AA  | Frequency  | Count
-----------------------------------
124    | P   | 1.0000     | 306  
109    | I   | 1.0000     | 303  
99     | N   | 1.0000     | 152  
161    | I   | 1.0000     | 131  
84     | I   | 1.0000     | 130  
277    | I   | 1.0000     | 87   




In [53]:
#Known VRC01 escape mutation
focus_residues = {
    276: 'N',
    279: 'K',
    281: 'T',
}

class_label = info_df["Label"].values
class_1_mask = class_label == 1
class_0_mask = class_label == 0

attr_array = arr_dict["Class 1 Attribution"].astype(float)
mean_attr_array = np.mean(attr_array, axis=0)

seq_1 = aligned_sequences[class_1_mask]
mean_attr_1 = mean_attr_array[class_1_mask]

seq_0 = aligned_sequences[class_0_mask]
mean_attr_0 = mean_attr_array[class_0_mask]

correlation_results = []
for mut_resno, mut_aa in focus_residues.items():
    resno_pos_arr = np.where(resno_array == str(mut_resno))[0]
    resno_pos = resno_pos_arr[0]

    # Get samples for all groups
    mut_mask = aligned_sequences[:, resno_pos].reshape(-1) == mut_aa
    mut_samples_all = mean_attr_array[mut_mask]
    mut_1_mask = seq_1[:, resno_pos].reshape(-1) == mut_aa
    mut_1_samples = mean_attr_1[mut_1_mask]
    mut_0_mask = seq_0[:, resno_pos].reshape(-1) == mut_aa
    mut_0_samples = mean_attr_0[mut_0_mask]
    
    n_all = len(mut_samples_all)
    n_1 = len(mut_1_samples)
    n_0 = len(mut_0_samples)

    # Calculate Correlation Distributions
    pairs_all = get_internal_correlation_pairs(mut_samples_all)
    pairs_1 = get_internal_correlation_pairs(mut_1_samples)
    pairs_0 = get_internal_correlation_pairs(mut_0_samples)
    
    avg_corr_all = np.mean(pairs_all) if len(pairs_all) > 0 else np.nan
    avg_corr_1 = np.mean(pairs_1) if len(pairs_1) > 0 else np.nan
    avg_corr_0 = np.mean(pairs_0) if len(pairs_0) > 0 else np.nan

    correlation_results.append({
        'Mutation': f"{mut_resno}{mut_aa}",
        'Avg_Internal_Correlation_All': avg_corr_all,
        'Avg_Internal_Correlation_Class_1': avg_corr_1,
        'Avg_Internal_Correlation_Class_0': avg_corr_0,
        'n_All': n_all, 'n_Class1': n_1, 'n_Class0': n_0
    })
results_df = pd.DataFrame(correlation_results)
print("\n--- Summary of Attribution Profile Spearman's Correlation from Strains with Identical Mutations ---")
print(results_df.to_string())


--- Summary of Attribution Profile Spearman's Correlation from Strains with Identical Mutations ---
   Avg_Internal_Correlation_All  Avg_Internal_Correlation_Class_0  Avg_Internal_Correlation_Class_1 Mutation  n_All  n_Class0  n_Class1
0                      0.411725                          0.402893                          0.461375     276N     43        13        30
1                      0.360745                          0.360745                               NaN     279K      4         4         0
2                      0.401961                          0.440626                          0.387151     281T    125        93        32
